# 04 — Simulation playground

Interactive what-if scenarios using the zero-sum state-level swing engine.

Edit the `SWINGS` dictionary below and re-run.

For full pipeline reports, run the CLI versions instead:
```
python -m src.simulation.zero_sum_state_swing_simulator
python -m src.analysis.seat_flip_report
python -m src.analysis.state_swing_sensitivity
```

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from src import data_io
from src.simulation import zero_sum_state_swing_simulator as zs

results = data_io.load_candidate_results()

In [ ]:
# Edit me. Keys are (state, party), values are vote-share swing in percentage points.
SWINGS = {
    ("Uttar Pradesh", "BJP"): -3.0,
    ("Uttar Pradesh", "SP"):  +3.0,
    ("Maharashtra", "BJP"):  -2.0,
    ("Maharashtra", "INC"):  +1.0,
    ("Maharashtra", "SHSUBT"): +1.0,
}

# Swap in our scenario so the simulator module uses it.
zs.STATE_SWINGS = SWINGS

simulated = (
    results.groupby(["state", "constituency"]) 
           .apply(zs.apply_zero_sum_swing)
           .reset_index(level=["state", "constituency"])
           .reset_index(drop=True)
)

sim_winners = simulated[simulated["sim_rank"] == 1].copy()
sim_winners["party"].value_counts().head(20)

## Which seats flipped?

In [ ]:
actual_winners = (
    results[results["winner"] == True]
    [["state", "constituency", "party"]]
    .rename(columns={"party": "actual_party"})
)

flips = actual_winners.merge(
    sim_winners[["state", "constituency", "party", "sim_vote_share"]]
        .rename(columns={"party": "sim_party"}),
    on=["state", "constituency"],
)
flips["flipped"] = flips["actual_party"] != flips["sim_party"]
flips[flips["flipped"]].sort_values(["state", "constituency"])